# No error estimate — so can an ensemble stand in for one?

**Book:** §5.10, Figure 5.8 &nbsp;·&nbsp; `ch05/error_estimates_ensemble.ipynb`

A PINN gives you a residual, not an error bound. And we have now seen three times over that a
*small residual can accompany a large error*. So what can a practitioner actually do?

$$-u'' = f(x),\qquad u(0)=u(1)=0,$$

with $f$ chosen so the exact solution is $\sin\pi x$ **plus a narrow bump** at $x=0.7$ — a local
feature the network has to work to resolve.

**Remedy implemented:** train an **ensemble of 5 seeds** and use the pointwise spread $\sigma(x)$
as an error indicator. Then check it honestly, by correlating $\sigma$ against the *true* error
(which we know here, and would not know in practice).

The comparison against the residual is the interesting part — the residual is the quantity you'd
naively reach for, and it is **much** the worse indicator of the two.

In [ ]:
import numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

def g1(f, x): return torch.autograd.grad(f, x, torch.ones_like(f), create_graph=True)[0]
def mlp(s, seed=0):
    torch.manual_seed(seed); L = []
    for i in range(len(s)-1):
        L.append(nn.Linear(s[i], s[i+1]))
        if i < len(s)-2: L.append(nn.Tanh())
    return nn.Sequential(*L)
rel = lambda p, e: float(np.sqrt(np.mean((p-e)**2)/np.mean((e-e.mean())**2 + 1e-30)))
M = {}

In [ ]:
A, SB, XB = 0.30, 0.05, 0.70
uex = lambda x: np.sin(np.pi*x) + A*np.exp(-((x-XB)/SB)**2)
def f_of(x):                                          # f = -u''  (analytic)
    z = (x - XB)/SB
    upp = -np.pi**2*np.sin(np.pi*x) + A*np.exp(-z**2)*(4*z**2 - 2)/SB**2
    return -upp
xg2 = np.linspace(0, 1, 400); u_ex = uex(xg2)
def poisson(seed):
    net = mlp([1,64,64,64,1], seed)
    U = lambda x: x*(1-x)*net(x)                      # hard Dirichlet
    opt = torch.optim.Adam(net.parameters(), 2e-3)
    for e in range(6000):
        if e == 4500:
            for g in opt.param_groups: g['lr'] = 4e-4
        opt.zero_grad()
        x = torch.rand(1024,1).requires_grad_(True)
        fx = torch.tensor(f_of(x.detach().numpy()), dtype=torch.float32)
        ((-g1(g1(U(x),x),x) - fx)**2).mean().backward(); opt.step()
    xt = torch.tensor(xg2, dtype=torch.float32).reshape(-1,1).requires_grad_(True)
    u = U(xt); r = (-g1(g1(u,xt),xt) - torch.tensor(f_of(xg2), dtype=torch.float32).reshape(-1,1))
    return u.detach().numpy().ravel(), np.abs(r.detach().numpy().ravel())
ENS = [poisson(s) for s in range(5)]
U = np.stack([e[0] for e in ENS]); RES = np.stack([e[1] for e in ENS])
umean, ustd = U.mean(0), U.std(0)
err = np.abs(umean - u_ex); resid = RES.mean(0)
M['ens_err'] = rel(umean, u_ex)
M['r_std_err']   = float(np.corrcoef(ustd, err)[0,1])
M['r_resid_err'] = float(np.corrcoef(resid, err)[0,1])
print(f'[5.10] ensemble-mean rel L2 = {M["ens_err"]:.2e}')
print(f'[5.10] corr(seed spread, true error) = {M["r_std_err"]:+.2f}')
print(f'[5.10] corr(residual,    true error) = {M["r_resid_err"]:+.2f}')

fig, ax = plt.subplots(1, 2, figsize=(12.4, 4.3))
ax[0].plot(xg2, u_ex, 'g', lw=2.8, alpha=.6, label='exact')
ax[0].plot(xg2, umean, 'r--', lw=1.6, label=f'ensemble mean, 5 seeds ({M["ens_err"]:.1e})')
ax[0].fill_between(xg2, umean-3*ustd, umean+3*ustd, color='tab:red', alpha=.35,
                   label=f'$\\pm 3\\sigma$ across seeds ($\\sigma\\sim${ustd.mean():.0e})')
ax[0].set_xlabel('x'); ax[0].set_ylabel('u'); ax[0].grid(alpha=.3); ax[0].legend(fontsize=9)
ax[0].set_title(f'(a) Ensemble of 5 seeds recovers $u$.\nThe $\\pm3\\sigma$ band is too thin to see — see (b)',
                fontsize=10.5)
ax[1].semilogy(xg2, err+1e-12, 'k', lw=2.2, label='true error $|u-u_{exact}|$ (unknowable)')
ax[1].semilogy(xg2, ustd+1e-12, 'r--', lw=1.6,
               label=f'seed spread $\\sigma$  (corr {M["r_std_err"]:+.2f})')
ax[1].semilogy(xg2, resid/resid.max()*err.max()+1e-12, 'b:', lw=1.6,
               label=f'residual, rescaled  (corr {M["r_resid_err"]:+.2f})')
ax[1].set_ylim(1e-7, 3e-4)
ax[1].set_xlabel('x'); ax[1].grid(alpha=.3); ax[1].legend(fontsize=8.5, loc='lower center')
ax[1].set_title('(b) The spread is a usable error indicator.\nThe residual alone is not.',
                fontsize=10.5)
plt.tight_layout(); plt.show()